In [3]:
import time
import difflib
import pandas as pd
from tqdm import tqdm


In [4]:
sales = pd.read_csv("data/raw/video_games_sales.csv")
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  object 
 1   Platform         16719 non-null  object 
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  object 
 4   Publisher        16665 non-null  object 
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  User_Score       10015 non-null  object 
 13  User_Count       7590 non-null   float64
 14  Developer        10096 non-null  object 
 15  Rating           9950 non-null   object 
dtypes: float64(9), object(7)
memory usage: 2.0+ MB


## Fixing dev name

In [5]:
indie_devs = pd.read_csv("data/raw/indie_games_developers.csv")
other_devs = pd.read_csv("data/raw/video_games_developers.csv")
devs = pd.concat([indie_devs, other_devs[indie_devs.columns]])
devs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 886 entries, 0 to 685
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Developer        886 non-null    object
 1   City             869 non-null    object
 2   Autonomous area  443 non-null    object
 3   Country          886 non-null    object
 4   Notable games    802 non-null    object
 5   Notes            470 non-null    object
dtypes: object(6)
memory usage: 48.5+ KB


In [6]:
unmatched_developers = sales[~sales["Developer"].isin(devs["Developer"])]["Developer"].unique()
print(len(unmatched_developers))

1383


In [7]:
no_match = []
skipped = []
dev_list = devs["Developer"].values
for unmatched_dev in tqdm(unmatched_developers):
    if pd.notna(unmatched_dev) and unmatched_dev not in skipped:
        time.sleep(0.25)
        match = difflib.get_close_matches(unmatched_dev, dev_list, n=1, cutoff=0.6)
        try:
            if match[0] and input(unmatched_dev + " -> " + match[0]) == "y":
                sales.loc[sales["Developer"] == unmatched_dev, "Developer"] = match[0]
            else:
                skipped.append(unmatched_dev)
        except IndexError as e:
            no_match.append(unmatched_dev)

  0%|          | 0/1383 [00:00<?, ?it/s]

100%|██████████| 1383/1383 [47:17<00:00,  2.05s/it]  


In [ ]:
for skipped_dev in tqdm(skipped):
    time.sleep(0.25)
    match = difflib.get_close_matches(skipped_dev, dev_list, n=1, cutoff=0.75)
    try:
        if match[0] and input(skipped_dev + " -> " + match[0]) == "y":
            sales.loc[sales["Developer"] == skipped_dev, "Developer"] = match[0]
    except IndexError as e:
            no_match.append(unmatched_dev)

In [11]:
sales.to_csv("./data/processed/processed_video_game_sales.csv")